# NumPy and Pandas Exercise Set

This notebook practices NumPy array creation, indexing, broadcasting, and vectorized operations. It then loads and cleans the public Titanic CSV with Pandas.

In [ ]:
import numpy as np
import pandas as pd

## 1. NumPy exercises

Practice creating arrays, selecting values, applying compatible shapes through broadcasting, and performing calculations without explicit loops.

In [ ]:
numbers = np.arange(1, 13).reshape(3, 4)
fractions = np.linspace(0, 1, 5)
empty_scores = np.zeros(4)
default_weights = np.ones(4)

print(numbers)
print(fractions)

In [ ]:
first_value = numbers[0, 0]
middle_row = numbers[1, :]
last_column = numbers[:, -1]
even_values = numbers[numbers % 2 == 0]

column_offsets = np.array([0, 10, 20, 30])
broadcast_result = numbers + column_offsets
squared = numbers ** 2
normalized = (numbers - numbers.min()) / (numbers.max() - numbers.min())

print('First value:', first_value)
print('Middle row:', middle_row)
print('Last column:', last_column)
print('Even values:', even_values)
print('Broadcast result:\n', broadcast_result)
print('Squared:\n', squared)
print('Normalized:\n', normalized)

## 2. Load the public Titanic CSV

`read_csv` can load a local path or a URL. This dataset is hosted publicly on GitHub.

In [ ]:
titanic_url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
passengers = pd.read_csv(titanic_url)

print(passengers.head())
print('Shape:', passengers.shape)
print('Missing values:\n', passengers.isna().sum())

## 3. Clean and filter

We keep the original data unchanged and create an analysis-ready copy. Numeric missing ages use the median, missing embarkation ports use the most common value, and missing cabin values are labeled `Unknown`.

In [ ]:
cleaned = passengers.copy().drop_duplicates()
cleaned['Age'] = cleaned['Age'].fillna(cleaned['Age'].median())
cleaned['Embarked'] = cleaned['Embarked'].fillna(cleaned['Embarked'].mode()[0])
cleaned['Cabin'] = cleaned['Cabin'].fillna('Unknown')
cleaned['Fare'] = pd.to_numeric(cleaned['Fare'], errors='coerce').fillna(0)
cleaned['Sex'] = cleaned['Sex'].str.strip().str.lower()

adult_survivors = cleaned.loc[(cleaned['Age'] >= 18) & (cleaned['Survived'] == 1)]
print(adult_survivors[['Name', 'Age', 'Sex', 'Pclass']].head())
print('Remaining missing values:', cleaned.isna().sum().sum())

## 4. Groupby, merge, and pivot table

In [ ]:
survival_by_class = (
    cleaned.groupby('Pclass', as_index=False)
    .agg(passengers=('PassengerId', 'count'), survival_rate=('Survived', 'mean'))
)

class_labels = pd.DataFrame({
    'Pclass': [1, 2, 3],
    'class_name': ['First class', 'Second class', 'Third class']
})
class_summary = survival_by_class.merge(class_labels, on='Pclass', how='left')

survival_pivot = pd.pivot_table(
    cleaned,
    values='Survived',
    index='Sex',
    columns='Pclass',
    aggfunc='mean'
)

print('Groupby plus merge:\n', class_summary)
print('\nSurvival pivot table:\n', survival_pivot)

## 5. Reusable cleaning script

The function below turns a raw passenger-style DataFrame into a consistent analysis-ready table. In a project, it can be moved into a `.py` file and tested separately.

In [ ]:
def clean_passenger_data(raw_data):
    data = raw_data.copy().drop_duplicates()
    data['Age'] = pd.to_numeric(data['Age'], errors='coerce')
    data['Age'] = data['Age'].fillna(data['Age'].median())
    data['Fare'] = pd.to_numeric(data['Fare'], errors='coerce').fillna(0)
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
    data['Cabin'] = data['Cabin'].fillna('Unknown')
    data['Sex'] = data['Sex'].str.strip().str.lower()
    data['Name'] = data['Name'].str.strip()
    return data

analysis_ready = clean_passenger_data(passengers)
print(analysis_ready.dtypes)
print('Total missing values:', analysis_ready.isna().sum().sum())

## Practice tasks

1. Use NumPy to find the row sums of `numbers`.
2. Filter passengers who paid more than the median fare.
3. Add a `FamilySize` column using `SibSp + Parch + 1`.
4. Make a pivot table of average fare by sex and passenger class.
5. Change one cleaning rule in `clean_passenger_data` and explain why your choice is appropriate.

## Worked practice and raw-data cleaning

The next cell completes the practice tasks and demonstrates a compact cleaning script. The function removes duplicates, standardizes text, converts numeric fields, fills missing values, and returns a table ready for analysis.

In [ ]:
# Complete the practice tasks.
row_sums = numbers.sum(axis=1)
high_fare = cleaned.loc[cleaned["Fare"] > cleaned["Fare"].median()]
cleaned["FamilySize"] = cleaned["SibSp"] + cleaned["Parch"] + 1
average_fare_pivot = pd.pivot_table(
    cleaned,
    values="Fare",
    index="Sex",
    columns="Pclass",
    aggfunc="mean",
)

print("Row sums:", row_sums)
print("Passengers above median fare:", len(high_fare))
print("Average fare by sex and class:\n", average_fare_pivot)

# A compact cleaner for a messy raw sales dataset.
def clean_sales_data(raw_data):
    data = raw_data.copy().drop_duplicates()
    data.columns = data.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    data["product"] = data["product"].astype("string").str.strip().str.lower()
    data["units"] = pd.to_numeric(data["units"], errors="coerce").fillna(0).astype(int)
    data["price"] = pd.to_numeric(data["price"], errors="coerce")
    data["price"] = data["price"].fillna(data["price"].median())
    data["status"] = data["status"].astype("string").str.strip().str.lower()
    data["revenue"] = data["units"] * data["price"]
    return data

messy_sales = pd.DataFrame({
    " Product ": [" Notebook", "Pen", "Notebook", "MUG "],
    " Units ": ["12", "25", None, "8"],
    " Price ": ["4.50", "1.20", "bad", None],
    " Status ": [" COMPLETE ", "complete", "pending", "complete"],
})

analysis_ready_sales = clean_sales_data(messy_sales)
print("\nAnalysis-ready sales:\n", analysis_ready_sales)
print("\nMissing values:\n", analysis_ready_sales.isna().sum())

## Practice task answers

1. `numbers.sum(axis=1)` calculates one sum for each row.
2. Rows above the median fare are selected with a boolean condition.
3. `FamilySize` counts the passenger plus siblings/spouses and parents/children.
4. The pivot table compares average fares across sex and class.
5. For the messy sales data, invalid or missing units become `0`, while missing prices use the median price. This keeps the row available without inventing a quantity or an arbitrary price.

In [ ]:
# Display every practice result clearly.
print("1. NumPy row sums:", row_sums)
print("\n2. Rows above the median fare:")
print(high_fare[["Name", "Fare", "Pclass"]].head())
print("\n3. FamilySize examples:")
print(cleaned[["Name", "SibSp", "Parch", "FamilySize"]].head())
print("\n4. Average fare pivot:")
print(average_fare_pivot)
print("\n5. Cleaning result and missing-value count:")
print(analysis_ready_sales)
print("Missing values:", int(analysis_ready_sales.isna().sum().sum()))